# Generate Gene Transcription Dataset for Replicate 1 and Replicate 2 of Yulong's 2017 Cell Cycle Experiment

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd


In [2]:
import os

# First make a dataframe that contains the metadata and filepaths
# for the RNA-seq data. This will make things convenient
# for when we want to read from disk
path = '/Users/trung/Research/_archive/data/bam/cell_cycle/rna/'
rows = []
for filename in os.listdir(path):
    if filename.endswith('bam'):
        fil_spl = filename.split('_')
        row = {'replicate': fil_spl[2].replace('rep', ''), 
               'time': fil_spl[3], 'full_path': path + filename}
        rows.append(row)

bam_df = pd.DataFrame.from_records(rows)
bam_df['time'] = bam_df['time'].astype(int)
bam_df['replicate'] = bam_df['replicate'].astype(int)
bam_df = bam_df.sort_values(['replicate', 'time'])
bam_df

,replicate,time,full_path
14,1,0,/Users/trung/Research/_archive/data/bam/cell_c...
12,1,20,/Users/trung/Research/_archive/data/bam/cell_c...
13,1,30,/Users/trung/Research/_archive/data/bam/cell_c...
1,1,40,/Users/trung/Research/_archive/data/bam/cell_c...
3,1,50,/Users/trung/Research/_archive/data/bam/cell_c...
17,1,60,/Users/trung/Research/_archive/data/bam/cell_c...
25,1,70,/Users/trung/Research/_archive/data/bam/cell_c...
0,1,80,/Users/trung/Research/_archive/data/bam/cell_c...
9,1,90,/Users/trung/Research/_archive/data/bam/cell_c...
5,1,100,/Users/trung/Research/_archive/data/bam/cell_c...


In [3]:
from cc_src.sgd import read_nondubious_genes_dataset

# Here is the list of genes we will be using for all of our deconvolutions
orfs = read_nondubious_genes_dataset()
orfs.head(2)

,gene,chr,cat,start,stop,strand,classification,length,TSS,PAS,manually_curated,promoter_start,promoter_end,gene_body_start,gene_body_end
orf_name,,,,,,,,,,,,,,,
YAL068C,PAU8,1,gene,1807,2169,-,Verified,362,2169,NaN,NaN,2169.0,2469.0,1669.0,2169.0
YAL067W-A,YAL067W-A,1,gene,2480,2707,+,Uncharacterized,227,2480,NaN,NaN,2180.0,2480.0,2480.0,2980.0


In [4]:
from src.timer import Timer
from cc_src.read_bam import read_rna_bam
from cc_src.transcription import calculate_read_counts
from cc_src.transcription import convert_to_TPM_all_times
    
def get_read_counts_TPM(replicate_bam):

    timer = Timer()
    all_times_read_counts = orfs[[]].copy()

    for _, row in replicate_bam.iterrows():

        time = row.time
        print(f"Reading BAM file for time {time} minutes")

        rna_reads = read_rna_bam(row.full_path, time, timer, log=True)
        orf_reads = calculate_read_counts(orfs, rna_reads)
        all_times_read_counts.loc[:, time] = orf_reads

        print(f"Done. {timer.get_time()}")
    
    TPMs = convert_to_TPM_all_times(all_times_read_counts, orfs['length'])

    return all_times_read_counts, TPMs

In [5]:
replicate_bam = bam_df[bam_df.replicate == 1]
rep1_read_counts, rep1_TPMs = get_read_counts_TPM(replicate_bam)


Reading BAM file for time 0 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:01:38.34
Reading BAM file for time 20 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:03:37.49
Reading BAM file for time 30 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:05:18.28
Reading BAM file for time 40 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:07:06.15
Reading BAM file for time 50 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:09:03.10
Reading BAM file for time 60 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:10:56.77
Reading BAM file for time 70 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:12:47.32
Reading BAM file for time 80 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:14:43.23
Reading BAM file for time 90 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:16:31.32
Reading BAM

In [6]:
rep1_TPMs

,0,20,30,40,50,60,70,80,90,100,110,120,130,140,150
orf_name,,,,,,,,,,,,,,,
YAL068C,0.119821,0.097373,0.000000,0.105016,0.095689,0.000000,0.000000,0.000000,0.000000,0.000000,0.069473,0.000000,0.000000,0.090285,0.087220
YAL067W-A,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
YAL067C,6.405208,3.008325,8.664824,14.578781,6.126558,2.829956,2.276200,1.727605,2.115023,2.488987,2.372318,3.461472,3.930270,3.449987,2.960575
YAL065C,0.561855,0.273955,0.315627,0.098487,0.179479,0.273931,0.000000,0.000000,0.000000,0.422212,0.065154,0.211539,0.078503,0.169343,0.000000
YAL064W-B,2.739486,3.339366,2.030533,1.700708,1.914281,2.226052,2.454628,1.799336,2.076963,1.029308,2.316392,2.363670,2.312537,1.806172,2.160296
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YPR200C,10.622496,9.981183,10.256264,7.758332,5.920503,6.024158,5.216566,5.581614,5.674078,5.903641,5.389156,6.249036,7.498251,6.503273,7.329575
YPR201W,3.072707,3.164849,2.341636,2.880936,2.254127,1.538738,1.743529,1.605174,1.950361,2.443264,2.775963,2.051441,2.271418,2.207591,2.912881
YPR202W,0.953853,1.550297,0.893059,0.923996,0.801837,1.101432,1.453283,1.741024,0.913479,1.697643,1.368081,1.417605,1.753600,2.004867,1.680999


In [7]:
replicate_bam = bam_df[bam_df.replicate == 2]
rep2_read_counts, rep2_TPMs = get_read_counts_TPM(replicate_bam)

rep2_TPMs

Reading BAM file for time 0 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:01:15.43
Reading BAM file for time 10 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:02:16.96
Reading BAM file for time 20 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:03:51.47
Reading BAM file for time 30 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:05:19.48
Reading BAM file for time 40 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:06:47.81
Reading BAM file for time 50 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:08:25.34
Reading BAM file for time 60 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:09:18.05
Reading BAM file for time 70 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:11:42.01
Reading BAM file for time 80 minutes
1..2..3..4..5..6..7..8..9..10..11..12..13..14..15..16..Done. 00:13:10.96
Reading BAM

,0,10,20,30,40,50,60,70,80,90,100,120,130,140
orf_name,,,,,,,,,,,,,,
YAL068C,0.157956,0.201953,0.129050,0.000000,0.276007,0.000000,0.000000,0.162258,0.130271,0.000000,0.000000,0.104494,0.000000,0.000000
YAL067W-A,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
YAL067C,9.984843,4.638454,6.295262,20.866170,19.326519,5.038601,2.792352,2.671388,2.012366,2.756863,2.308499,3.292074,3.175976,3.359004
YAL065C,1.036945,0.378793,0.242052,0.000000,0.129423,0.000000,0.000000,0.228255,0.366515,0.212002,0.000000,0.195995,0.098348,0.000000
YAL064W-B,2.558056,3.270575,3.933987,2.548879,2.892261,1.930082,2.145461,1.313867,2.854313,2.691871,2.297961,2.886797,2.897139,3.465467
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YPR200C,7.439246,9.511365,7.627117,6.762331,7.519073,6.713582,6.655307,6.368231,6.255681,5.532070,5.754674,5.693348,8.522195,7.774568
YPR201W,2.307928,3.613194,2.886085,3.695244,2.715961,2.629805,3.626430,2.370791,2.641481,2.527787,2.547507,2.212287,2.689271,2.138492
YPR202W,1.257432,2.030749,1.730226,0.767024,1.098596,1.597890,2.075932,2.753323,1.692019,1.041854,1.473905,1.313437,1.669647,1.698354


In [42]:
rep1_TPMs.head(2)

,0,20,30,40,50,60,70,80,90,100,110,120,130,140,150
orf_name,,,,,,,,,,,,,,,
YAL068C,0.119821,0.097373,0.0,0.105016,0.095689,0.0,0.0,0.0,0.0,0.0,0.069473,0.0,0.0,0.090285,0.08722
YAL067W-A,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.00000


In [43]:
rep2_TPMs.head(2)

,0,10,20,30,40,50,60,70,80,90,100,120,130,140
orf_name,,,,,,,,,,,,,,
YAL068C,0.157956,0.201953,0.12905,0.0,0.276007,0.0,0.0,0.162258,0.130271,0.0,0.0,0.104494,0.0,0.0
YAL067W-A,0.000000,0.000000,0.00000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0


In [47]:
# Save each to disk
rep1_TPMs.to_csv('datasets/yl_cell_cycle/replicate1_gene_expression.csv')
rep2_TPMs.to_csv('datasets/yl_cell_cycle/replicate2_gene_expression.csv')